In [1]:
%pwd

'/content'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls

drive  sample_data


In [4]:
%cd /content/
!ls

/content
drive  sample_data


In [5]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 49.9 MB/s eta 0:00:00


In [6]:
# Logically Conclusive and Inconclusive KBs

import json
import os
import time
import datetime
from pathlib import Path


PROMPT = """
1.  Select a polysemy noun. Each one of the polysemy noun keywords in - bat, head, nail, paper- that have two different meanings in the noun forms and with real world references. Example: bat. Add this to json key: "KB".
2. Select one fact each for each one of the two meanings for the each one of the keywords. Example: "Bat is a mammal. Bat is used in baseball." where this information is valid in real world. Add this to json key: "KB".
3. Select two distinct features of two different meanings of a given keyword and generate two new facts using "have" or "contain" depending on reference. Example: "Bats have a barrel. Bat flies with wings". Add this to json key: "KB".
4. Generate two equivalence clauses for two distinct features selected in previous step, for the two meanings of a given keyword. \textbf{(for ensuring meaning preservation by common sense, not refutation)}. Example: "Wings are not barrels. Barrels are not wings.".  Add this to json key: "KB".
5. Introduce a repeated re-assertion of a fact generated in step 3 with "can" or other relevant forms. Add this to json key: "KB".
6. Select a random color. Prepend the color to new instance of keyword and generate a query using this new colored instance to construct an introduction. For the query, use one of the two facts to replace the keyword with colored keyword instance $\textit{color} \in \textit{black, white, brown, red, yellow, blue, green, pink} $. Example: "There is a Brown bat" or "There is a pink nail" and so on. Add this to json key: "KB".
7. Add an action or negated action sentence based on step 2, replacing the noun with colored keyword instance (Ex: brown bat). The idea is to add confusion such that the action/property that could distinguish two meanings of the polysemy noun is absent, leading to inconclusiveness. Examples: "There is a brown bat. Brown bat flew." or "There is a pink nail. Pink nail did not fasten the object." Add this to json key: "KB".
8. Using the facts in step 1 such as containment or $\exists$ assertions from facts, generate a query as a statement. Example: "Query: Brown bat is a mammal", or "Query: Pink nail is part of the finger". The resulting query "Is the statement "Brown bat is a mammal" true or false or inconclusive?".
9. Generate a question with multiple choices can be framed as: "For the statement: Brown bat is a mammal, select the Correct Ans from following choices: 1. True, 2. False, 3. Inconclusive, 4. True if brown bat is a mammal and Inconclusive i.e. undecidable if brown bat could have accidentally thrown by a baseball player.".
10. Place results from steps 8 and 9 into a json list with key: "Inconclusive".
11. Generate reasoning why this KB is "Inconclusive".
12. Now combine the resulting KB in this format: {"key": "Bat", "kb": "Bat is a mammal. Bat is used in baseball. Bats have a barrel. Bat flies with wings. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew." "inconclusive": ["query": "Brown bat is a mammal.", "mcq": "For the statement: Brown bat is a mammal, select the Correct Ans from following choices: 1. True, 2. False, 3. Inconclusive, 4. True if brown bat is a mammal and Inconclusive i.e. undecidable if brown bat could have accidentally thrown by a baseball player.", "Reasoning": "Brown bat can be a mammal or baseball bat when a baseball player may have thrown the brown bat."]}
13. For each generated KB, generate a conclusive query, mcq, reasoning format and add a "conclusive" list with query, mcq, reasoning items and add it to the array.
14. Repeat all the steps for remaining polysemy noun keywords listed in step 1.
15. Generate other polysemy noun keywords. And repeat steps 1 to 10 for each new polysemy noun.
16. Place all generated results in a json array format.
"""





<>:16: SyntaxWarning: invalid escape sequence '\i'
<>:16: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_563/3091799259.py:16: SyntaxWarning: invalid escape sequence '\i'
  6. Select a random color. Prepend the color to new instance of keyword and generate a query using this new colored instance to construct an introduction. For the query, use one of the two facts to replace the keyword with colored keyword instance $\textit{color} \in \textit{black, white, brown, red, yellow, blue, green, pink} $. Example: "There is a Brown bat" or "There is a pink nail" and so on. Add this to json key: "KB".


## Gemini Pro 2.5 Flash KB Generation for Conclusive and Inconclusive

In [7]:
!pip install google-generativeai
!pip show google-generativeai

Name: google-generativeai
Version: 0.8.6
Summary: Google Generative AI High level API client library and tools.
Home-page: https://github.com/google/generative-ai-python
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: google-ai-generativelanguage, google-api-core, google-api-python-client, google-auth, protobuf, pydantic, tqdm, typing-extensions
Required-by: 


In [8]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [9]:
from google import generativeai, genai

client = genai.Client(api_key = GEMINI_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [10]:
MODEL_ID = "gemini-2.5-flash"

In [11]:
model_info = client.models.get(model=MODEL_ID)
print(model_info)

name='models/gemini-2.5-flash' display_name='Gemini 2.5 Flash' description='Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.' version='001' endpoints=None labels=None tuned_model_info=TunedModelInfo() input_token_limit=1048576 output_token_limit=65536 supported_actions=['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent'] default_checkpoint_id=None checkpoints=None temperature=1.0 max_temperature=2.0 top_p=0.95 top_k=64 thinking=True


In [ ]:
response = client.models.generate_content(
    model = MODEL_ID,
    contents=PROMPT,
    config = genai.types.GenerateContentConfig(temperature=0.7)
)
result = response.text

In [ ]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""```json
[
  {
    "key": "Bat",
    "kb": "Bat is a mammal. Bat is used in baseball. Bat flies with wings. Bats have a barrel. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew.",
    "inconclusive": {
      "query": "Brown bat is a mammal.",
      "mcq": "For the statement: Brown bat is a mammal, select the Correct Ans from following choices: 1. True, 2. False, 3. Inconclusive, 4. True if brown bat is a mammal and Inconclusive i.e. undecidable if brown bat could have accidentally thrown by a baseball player.",
      "reasoning": "Brown bat can refer to the mammal or a brown-colored baseball bat. While 'flew' typically suggests the mammal, a baseball bat can also be thrown through the air, making the meaning ambiguous and the statement inconclusive without further clarifi

In [ ]:
results = result.replace("```json","").replace('\\"',"'").replace('\\"',"'").replace("```","")

In [ ]:
import json
json_list = json.loads(results)

In [ ]:
filename = "gemini_generated_kbs.jsonl"
file_path = os.path.join('/content/', filename)

with open(file_path, "w", encoding="utf-8") as output_file:
    json.dump(json_list, output_file, indent=4)

assert os.path.exists(file_path), "File does not exist!"

In [37]:
import json

filename = "/content/gemini_generated_kbs.jsonl"
json_list = []
with open(filename, 'r') as file:
    json_list = json.load(file)


In [38]:

json_list.append("\n For each KB in above json list, combine KB, rename json key inconclusive as neutral query and conclusive as 'not neutral' and modify the query question as neutral or not neutral?  and modify mcq as 1. Neutral 2. Not neutral. 3. Neutral if this and not neutral if that and generate new json list.")
PROMPT2 = json.dumps(json_list)


In [39]:
response = client.models.generate_content(
    model = MODEL_ID,
    contents=PROMPT2,
    config = genai.types.GenerateContentConfig(temperature=0.7)
)
result = response.text

In [41]:
print(result)

```json
[
  {
    "key": "Bat",
    "KB": "Bat is a mammal. Bat is used in baseball. Bat flies with wings. Bats have a barrel. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew.",
    "neutral_query": [
      {
        "query": "Is the statement 'Brown bat is a mammal' neutral or not neutral?",
        "mcq": "For the statement: Brown bat is a mammal, select the Correct Ans from following choices: 1. Neutral, 2. Not neutral, 3. Neutral if brown bat could have accidentally thrown by a baseball player and Not neutral if brown bat is a mammal.",
        "Reasoning": "The term 'brown bat' is ambiguous. It could refer to a brown-colored flying mammal, which is a mammal. However, it could also refer to a baseball bat that is brown, which is not a mammal. The action 'flew' could apply to the mammal or a baseball bat that was thrown, thus it's inconclusive."
      }
    ],
    "not_neutral_query": [
      {
        "query": "Is the statement 'A br

In [42]:
results = result.replace("```json","").replace('\\"',"'").replace('\\"',"'").replace("```","")

In [43]:
results

'\n[\n  {\n    "key": "Bat",\n    "KB": "Bat is a mammal. Bat is used in baseball. Bat flies with wings. Bats have a barrel. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew.",\n    "neutral_query": [\n      {\n        "query": "Is the statement \'Brown bat is a mammal\' neutral or not neutral?",\n        "mcq": "For the statement: Brown bat is a mammal, select the Correct Ans from following choices: 1. Neutral, 2. Not neutral, 3. Neutral if brown bat could have accidentally thrown by a baseball player and Not neutral if brown bat is a mammal.",\n        "Reasoning": "The term \'brown bat\' is ambiguous. It could refer to a brown-colored flying mammal, which is a mammal. However, it could also refer to a baseball bat that is brown, which is not a mammal. The action \'flew\' could apply to the mammal or a baseball bat that was thrown, thus it\'s inconclusive."\n      }\n    ],\n    "not_neutral_query": [\n      {\n        "query": "Is the 

In [44]:
import json

filename = "gemini_generated_kbs_nli.jsonl"
json_list = json.loads(results)
file_path = os.path.join('/content/', filename)

with open(file_path, "w", encoding="utf-8") as output_file:
    json.dump(json_list, output_file, indent=4)

assert os.path.exists(file_path), "File does not exist!"

## Claude KB Generation for Conclusive and Inconclusive

In [45]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 11.4 MB/s eta 0:00:00


In [46]:
from google.colab import userdata
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

In [47]:
import anthropic

client = anthropic.Anthropic(api_key = ANTHROPIC_API_KEY)
print(client)

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens = 10000,
    messages=[
        {
            "role": "user",
            "content": PROMPT,
        }
    ],
)
print(message.content)

[TextBlock(citations=None, text='```json\n[\n  {\n    "key": "bat",\n    "kb": "Bat is a mammal. Bat is used in baseball. Bats have wings. Bats have a barrel. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew.",\n    "inconclusive": {\n      "query": "Brown bat is a mammal.",\n      "mcq": "For the statement: Brown bat is a mammal, select the Correct Answer from following choices: 1. True, 2. False, 3. Inconclusive, 4. True if brown bat is a mammal and Inconclusive i.e. undecidable if brown bat could have accidentally been thrown by a baseball player.",\n      "reasoning": "The statement is inconclusive because \'brown bat\' could refer to either a brown-colored mammal that flies, or a brown-colored baseball bat that was thrown. The fact that \'brown bat flew\' doesn\'t definitively distinguish between the two meanings since a baseball bat can also \'fly\' through the air when thrown."\n    },\n    "conclusive": {\n      "query": "Brown ba

In [ ]:
result = message.content[0].text.replace("```json","").replace("```","")

In [ ]:
result

'\n[\n  {\n    "key": "bat",\n    "kb": "Bat is a mammal. Bat is used in baseball. Bats have wings. Bats have a barrel. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew.",\n    "inconclusive": {\n      "query": "Brown bat is a mammal.",\n      "mcq": "For the statement: Brown bat is a mammal, select the Correct Answer from following choices: 1. True, 2. False, 3. Inconclusive, 4. True if brown bat is a mammal and Inconclusive i.e. undecidable if brown bat could have accidentally been thrown by a baseball player.",\n      "reasoning": "The statement is inconclusive because \'brown bat\' could refer to either a brown-colored mammal that flies, or a brown-colored baseball bat that was thrown. The fact that \'brown bat flew\' doesn\'t definitively distinguish between the two meanings since a baseball bat can also \'fly\' through the air when thrown."\n    },\n    "conclusive": {\n      "query": "Brown bat has wings.",\n      "mcq": "For the s

In [ ]:
import json
json_list = json.loads(result)

In [ ]:
filename = "claude_generated_kbs.jsonl"
file_path = os.path.join('/content/', filename)

with open(file_path, "w", encoding="utf-8") as output_file:
    json.dump(json_list, output_file, indent=4)

assert os.path.exists(file_path), "File does not exist!"

In [48]:
filename = "/content/claude_generated_kbs.jsonl"
json_list = []
with open(filename, 'r') as file:
    json_list = json.load(file)


json_list.append("\n For each KB in above json list, combine KB, rename json key inconclusive as neutral query and conclusive as 'not neutral' and modify the query question as neutral or not neutral?  and modify mcq as 1. Neutral 2. Not neutral. 3. Neutral if this and not neutral if that and generate new json list.")
PROMPT2 = json.dumps(json_list)

In [49]:
message = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens = 10000,
    messages=[
        {
            "role": "user",
            "content": PROMPT2,
        }
    ],
)
print(message.content)

[TextBlock(citations=None, text='```json\n[\n  {\n    "key": "bat",\n    "kb": "Bat is a mammal. Bat is used in baseball. Bats have wings. Bats have a barrel. Wings are not barrels. Barrels are not wings. Bats can fly. There is a brown bat. Brown bat flew.",\n    "neutral_query": {\n      "query": "Is \'Brown bat is a mammal\' neutral or not neutral?",\n      "mcq": "For the statement: Brown bat is a mammal, select the Correct Answer from following choices: 1. Neutral, 2. Not neutral, 3. Neutral if brown bat could refer to either the mammal or baseball equipment, and Not neutral if the context definitively identifies which meaning is intended.",\n      "reasoning": "The statement is neutral because \'brown bat\' could refer to either a brown-colored mammal that flies, or a brown-colored baseball bat that was thrown. The fact that \'brown bat flew\' doesn\'t definitively distinguish between the two meanings since a baseball bat can also \'fly\' through the air when thrown."\n    },\n   

In [50]:
result = message.content[0].text.replace("```json","").replace("```","")

In [51]:
import json
json_list = json.loads(result)

In [52]:
filename = "claude_generated_kbs_nli.jsonl"
file_path = os.path.join('/content/', filename)

with open(file_path, "w", encoding="utf-8") as output_file:
    json.dump(json_list, output_file, indent=4)

assert os.path.exists(file_path), "File does not exist!"